# Cancer Drug Sensitivity Prediction (GDSC)

XGBoost regression that predicts drug sensitivity (LN_IC50) for 242,035 drug and cell line pairs from the Genomics of Drug Sensitivity in Cancer (GDSC) data, with SHAP to explain the model.

**Leakage warning:** AUC and Z_SCORE are used as features, but both come from the same dose-response curve as LN_IC50. The reported R² of 0.994 is therefore inflated. See the README.

Place the data file as `genomic_data.csv` next to this notebook (see `data/README.md`).

### Load the Dataset

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("genomic_data.csv")  # Update with path if necessary

### Exploratory Analysis

In [ ]:
# Display information about the dataset
df.info()

# Preview top rows
df.head()

### Handling Missing Values

In [ ]:
# 🔍 Check for missing values
missing_counts = df.isnull().sum()
missing_percentage = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_counts, 'Percent Missing': missing_percentage})
missing_df[missing_df['Missing Values'] > 0].sort_values(by="Missing Values", ascending=False)


# 🔧 Columns to fill missing values with 'Unknown'
fill_unknown_cols = [
    'TCGA_DESC',
    'GDSC Tissue descriptor 1',
    'GDSC Tissue descriptor 2',
    'Cancer Type (matching TCGA label)',
    'Microsatellite instability Status (MSI)',
    'Screen Medium',
    'Growth Properties',
    'CNA',
    'Gene Expression',
    'Methylation',
    'TARGET'
]

# 🔄 Fill missing values in-place
df[fill_unknown_cols] = df[fill_unknown_cols].fillna('Unknown')

# 🧹 Drop rows where TARGET_PATHWAY is missing (if any)
df.dropna(subset=['TARGET_PATHWAY'], inplace=True)

# ✅ Confirm all missing values are handled
print("Remaining missing values:\n", df.isnull().sum())

### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 📉 Distribution of LN_IC50, AUC, and Z_SCORE
plt.figure(figsize=(18, 5))

# LN_IC50
plt.subplot(1, 3, 1)
sns.histplot(df['LN_IC50'], kde=True, bins=50, color='skyblue')
plt.title("LN_IC50 Distribution")

# AUC
plt.subplot(1, 3, 2)
sns.histplot(df['AUC'], kde=True, bins=50, color='salmon')
plt.title("AUC Distribution")

# Z_SCORE
plt.subplot(1, 3, 3)
sns.histplot(df['Z_SCORE'], kde=True, bins=50, color='seagreen')
plt.title("Z_SCORE Distribution")

plt.tight_layout()
plt.show()

### Distribution of top 10 tissue types (R) and top 10 most sensitive drugs (L)

In [ ]:
# Get top 10 tissue types
top_tissues = df['GDSC Tissue descriptor 1'].value_counts().head(10)
# Plot top 10 tissue types
plt.figure(figsize=(10, 6))
sns.barplot(x=top_tissues.values, y=top_tissues.index)
plt.title("Top 10 Tissue Types Represented")
plt.xlabel("Number of Records")
plt.ylabel("Tissue Type")
plt.tight_layout()
plt.show()

# Get top 10 most sensitive drugs by lowest average LN_IC50
sensitive_drugs = df.groupby('DRUG_NAME')['LN_IC50'].mean().sort_values().head(10)
# Plot top 10 most sensitive drugs
plt.figure(figsize=(10, 6))
sns.barplot(x=sensitive_drugs.values, y=sensitive_drugs.index)
plt.title("Top 10 Most Sensitive Drugs (Lowest Mean LN_IC50)")
plt.xlabel("Mean LN_IC50")
plt.ylabel("Drug Name")
plt.tight_layout()
plt.show()

### 📦LN_IC50 Distribution Across Top 10 Tissue Types

In [ ]:
# Boxplot of LN_IC50 across top 10 tissue types
top_tissues = df['GDSC Tissue descriptor 1'].value_counts().head(10).index
subset = df[df['GDSC Tissue descriptor 1'].isin(top_tissues)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=subset, x='GDSC Tissue descriptor 1', y='LN_IC50')
plt.title('LN_IC50 Distribution by Tissue Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 🎯LN_IC50 Distribution Across Top Drug Target Pathways

In [ ]:
#Boxplot of LN_IC50 Distribution across Top Target Pathways
top_pathways = df['TARGET_PATHWAY'].value_counts().head(10).index
subset_pathways = df[df['TARGET_PATHWAY'].isin(top_pathways)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=subset_pathways, x='TARGET_PATHWAY', y='LN_IC50')
plt.xticks(rotation=45)
plt.title('LN_IC50 Distribution Across Top Target Pathways')
plt.tight_layout()
plt.savefig('LN_IC50 Distribution across Top Target Pathways.png', dpi=300, bbox_inches='tight')
plt.show()

### 🧪 Number of Unique Drugs Tested per Cell Line

In [ ]:
# Calculate number of unique drugs tested per cell line
drug_counts_per_cell_line = df.groupby('CELL_LINE_NAME')['DRUG_NAME'].nunique().sort_values(ascending=False).head(15)

# Plotting
plt.figure(figsize=(10, 6))
sns.barplot(x=drug_counts_per_cell_line.values, y=drug_counts_per_cell_line.index)
plt.title("Number of Unique Drugs Tested per Cell Line")
plt.xlabel("Drug Count")
plt.ylabel("Cell Line Name")
plt.tight_layout()
plt.show()

### Prediction Analysis

> **Note on this cell.** This code is copied as it appears in my original write-up. It still has gaps from an earlier draft: it refers to `gdsc_df` and `gdsc_df_clean` (the loaded data is called `df` above), and `lower_bound` and `upper_bound` for the IQR clipping are never defined. Set those names to match your session before running it. AUC and Z_SCORE are still in the feature set here, which causes target leakage (see README).

In [ ]:
# 📦 Imports
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# 🧹 Clean missing values in case we missed something
df_clean = gdsc_df.dropna(subset=['TARGET'])

df_clean['Cancer Type (matching TCGA label)'].fillna(gdsc_df_clean['Cancer Type (matching TCGA label)'].mode()[0], inplace=True)
gdsc_df_clean['Microsatellite instability Status (MSI)'].fillna(gdsc_df_clean['Microsatellite instability Status (MSI)'].mode()[0], inplace=True)

# Apply one-hot encoding to key categorical features
encoded_df = pd.get_dummies(gdsc_df_clean, columns=['DRUG_NAME', 'TARGET_PATHWAY', 'Cancer Type (matching TCGA label)'], drop_first=True)

# Cap outliers in LN_IC50 using IQR method (Winsorization)
Q1 = gdsc_df_clean['LN_IC50'].quantile(0.25)
Q3 = gdsc_df_clean['LN_IC50'].quantile(0.75)

df_clean['LN_IC50'] = gdsc_df_clean['LN_IC50'].clip(lower=lower_bound, upper=upper_bound)

# Model Training
# Features and target
X = gdsc_df_clean.drop(['LN_IC50', 'COSMIC_ID', 'CELL_LINE_NAME'], axis=1)  # Dropping IDs
y = gdsc_df_clean['LN_IC50']

# One-hot encode remaining categorical features
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

# Evaluation Metrics
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

### 🔍 SHAP Analysis: Global Feature Importance

> **Note.** The step that creates `shap_values` (for example `explainer = shap.Explainer(model)` then `shap_values = explainer(X_test)`) is missing from the original write-up, so define it before running the two SHAP cells.

In [ ]:
import shap

# SHAP summary plot for overall feature impact
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test)
plt.title("SHAP Summary Plot - Global Feature Importance")
plt.show()

### 📊 SHAP Bar Plot: Top Influential Features

In [ ]:
# SHAP beeswarm plot (shows impact and distribution)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar")
plt.title("SHAP Bar Plot - Top Influential Features")
plt.show()

### 🔧 Hyperparameter Tuning with XGBoost and RandomizedSearchCV

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

xgb_model = XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42)

param_dist = {
    'n_estimators': [300, 500, 700],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],# L1 regularization
    'reg_lambda': [0.5, 1, 1.5]  # L2 regularization
}

random_search = RandomizedSearchCV(estimator=xgb_model,
                                   param_distributions=param_dist,
                                   n_iter=20,  # Number of random combinations
                                   scoring='neg_root_mean_squared_error',
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
random_search.fit(X_train, y_train)
                          

In [ ]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

# Evaluation Metrics
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)

print("Best Parameters Found:", random_search.best_params_)
print(f"Tuned R² Score:{r2:.4f}")
print(f"Tuned RMSE:{rmse:.4f}")
print(f"Tuned MAE:{mae:.4f}")